验证四个关键修复点的逻辑正确性

In [1]:

import numpy as np

# ── 验证修复 #1：_map_or_zeros ──────────────────────────────────────
def _map_or_zeros(m, shape_hw):
    if m is None:
        return np.zeros(shape_hw, dtype=np.float32)
    return np.nan_to_num(np.asarray(m, dtype=np.float32), nan=0.0)

shape = (64, 64)
assert _map_or_zeros(None, shape).shape == shape
assert _map_or_zeros(None, shape).sum() == 0
arr = np.full(shape, np.nan, dtype=np.float32)
assert _map_or_zeros(arr, shape).sum() == 0
print("修复 #1 [OK]  None sustained_map → 全零数组，不再 AttributeError")

# ── 验证修复 #3：精确路径匹配 ────────────────────────────────────────
all_trials = [
    "/data/trial_1/trial_1_corrected_movie.tif",
    "/data/trial_10/trial_10_corrected_movie.tif",
    "/data/trial_2/trial_2_corrected_movie.tif",
]
import os
trial_dir_to_stat = {}
for p in all_trials:
    t_dir = os.path.dirname(p)
    t_name = os.path.basename(t_dir)
    trial_dir_to_stat[t_name] = os.path.join(t_dir, "suite2p", "plane0", "stat.npy")

# trial_1 不应匹配到 trial_10
assert trial_dir_to_stat.get("trial_1") == "/data/trial_1/suite2p/plane0/stat.npy"
assert trial_dir_to_stat.get("trial_10") == "/data/trial_10/suite2p/plane0/stat.npy"
# 旧版 "trial_1" in "/data/trial_10/..." 为 True，新版精确匹配不会混淆
print("修复 #3 [OK]  trial_1 不再错误匹配 trial_10")

# ── 验证修复 #7：子阈值 sigma 估计 ──────────────────────────────────
def noise_sigma_from_subthreshold(combined_map):
    arr = np.asarray(combined_map, dtype=np.float32)
    finite = arr[np.isfinite(arr)]
    if finite.size == 0:
        return 1.0
    med = float(np.median(finite))
    background = finite[finite <= med]
    if background.size < 10:
        return max(float(np.std(finite)), 1e-6)
    mad = float(np.median(np.abs(background - med)))
    sigma = 1.4826 * mad
    if sigma <= 0:
        sigma = float(np.std(finite))
    return max(float(sigma), 1e-6)

def robust_std_old(arr):
    arr = arr[np.isfinite(arr)]
    if arr.size == 0: return 0.0
    med = np.median(arr)
    mad = np.median(np.abs(arr - med))
    return float(1.4826 * mad)

# 模拟偏态 combined_map：95% 背景噪声 + 5% 强响应
rng = np.random.default_rng(42)
background = rng.normal(0, 1.0, 9500).astype(np.float32)   # 真实噪声 sigma=1
signal     = rng.normal(10, 1.0, 500).astype(np.float32)   # 响应像素
combined   = np.concatenate([background, signal])

sigma_old = robust_std_old(combined)
sigma_new = noise_sigma_from_subthreshold(combined)

print(f"\n修复 #7 [OK]  偏态 combined_map 的 sigma 估计对比（真实 sigma=1.0）:")
print(f"  旧版全图 MAD sigma : {sigma_old:.4f}  （被响应像素拉高，偏大）")
print(f"  新版子阈值 sigma   : {sigma_new:.4f}  （接近真实背景噪声）")
assert sigma_new < sigma_old, "新版 sigma 应更接近背景噪声，不应大于旧版"

# ── 验证修复 #8：相对阈值 nostim 判断 ───────────────────────────────
NOSTIM_REL_THRESH = 0.05
def is_nostim(b_max, b_min):
    if b_max <= 0:
        return True
    return (b_max - b_min) / b_max < NOSTIM_REL_THRESH

# 旧版：b_max-b_min < 50（绝对值）
# 场景1：高增益相机，背景亮度 10000，刺激幅度 200 → 旧版不判 nostim，新版也不判
assert not is_nostim(10000, 9800)   # range=200, rel=2% < 5% → nostim（正确，信号弱）
assert not is_nostim(10000, 9000)   # range=1000, rel=10% → 有刺激（正确）
# 场景2：低增益相机，背景亮度 100，刺激幅度 30 → 旧版判 nostim（错误），新版不判
assert not is_nostim(100, 70)       # range=30, rel=30% → 有刺激（正确）
# 场景3：真正无刺激
assert is_nostim(100, 98)           # range=2, rel=2% → nostim（正确）
print("\n修复 #8 [OK]  相对阈值 nostim 判断，对不同相机/增益均鲁棒")

print("\n所有关键修复验证通过 ✓")


修复 #1 [OK]  None sustained_map → 全零数组，不再 AttributeError
修复 #3 [OK]  trial_1 不再错误匹配 trial_10

修复 #7 [OK]  偏态 combined_map 的 sigma 估计对比（真实 sigma=1.0）:
  旧版全图 MAD sigma : 1.0544  （被响应像素拉高，偏大）
  新版子阈值 sigma   : 1.0192  （接近真实背景噪声）
---------------------------------------------------------------------------
AssertionError                            Traceback (most recent call last)
Cell In[1], line 81
     77     return (b_max - b_min) / b_max < NOSTIM_REL_THRESH
     79 # 旧版：b_max-b_min < 50（绝对值）
     80 # 场景1：高增益相机，背景亮度 10000，刺激幅度 200 → 旧版不判 nostim，新版也不判
---> 81 assert not is_nostim(10000, 9800)   # range=200, rel=2% < 5% → nostim（正确，信号弱）
     82 assert not is_nostim(10000, 9000)   # range=1000, rel=10% → 有刺激（正确）
     83 # 场景2：低增益相机，背景亮度 100，刺激幅度 30 → 旧版判 nostim（错误），新版不判

AssertionError: 

验证四个关键修复点的逻辑正确性（修正测试用例）

In [3]:

import numpy as np

# ── 验证修复 #1：_map_or_zeros ──────────────────────────────────────
def _map_or_zeros(m, shape_hw):
    if m is None:
        return np.zeros(shape_hw, dtype=np.float32)
    return np.nan_to_num(np.asarray(m, dtype=np.float32), nan=0.0)

shape = (64, 64)
assert _map_or_zeros(None, shape).shape == shape
assert _map_or_zeros(None, shape).sum() == 0
arr = np.full(shape, np.nan, dtype=np.float32)
assert _map_or_zeros(arr, shape).sum() == 0
print("修复 #1 [OK]  None sustained_map → 全零数组，不再 AttributeError")

# ── 验证修复 #3：精确路径匹配 ────────────────────────────────────────
import os
all_trials = [
    "/data/trial_1/trial_1_corrected_movie.tif",
    "/data/trial_10/trial_10_corrected_movie.tif",
    "/data/trial_2/trial_2_corrected_movie.tif",
]
trial_dir_to_stat = {}
for p in all_trials:
    t_dir = os.path.dirname(p)
    t_name = os.path.basename(t_dir)
    trial_dir_to_stat[t_name] = os.path.join(t_dir, "suite2p", "plane0", "stat.npy")

assert trial_dir_to_stat["trial_1"]  == "/data/trial_1/suite2p/plane0/stat.npy"
assert trial_dir_to_stat["trial_10"] == "/data/trial_10/suite2p/plane0/stat.npy"
# 旧版："trial_1" in "/data/trial_10/trial_10_corrected_movie.tif" → True（错误匹配）
# 新版：精确 key 查找，trial_1 和 trial_10 完全独立
print("修复 #3 [OK]  trial_1 不再错误匹配 trial_10")

# ── 验证修复 #7：子阈值 sigma 估计 ──────────────────────────────────
def noise_sigma_from_subthreshold(combined_map):
    arr = np.asarray(combined_map, dtype=np.float32)
    finite = arr[np.isfinite(arr)]
    if finite.size == 0:
        return 1.0
    med = float(np.median(finite))
    background = finite[finite <= med]
    if background.size < 10:
        return max(float(np.std(finite)), 1e-6)
    mad = float(np.median(np.abs(background - med)))
    sigma = 1.4826 * mad
    if sigma <= 0:
        sigma = float(np.std(finite))
    return max(float(sigma), 1e-6)

def robust_std_old(arr):
    arr = arr[np.isfinite(arr)]
    if arr.size == 0: return 0.0
    med = np.median(arr)
    mad = np.median(np.abs(arr - med))
    return float(1.4826 * mad)

# 模拟偏态 combined_map：95% 背景噪声(sigma=1) + 5% 强响应(均值=10)
# 这是逻辑验证用的合成数据，不是真实实验数据
rng = np.random.default_rng(42)
background = rng.normal(0, 1.0, 9500).astype(np.float32)
signal     = rng.normal(10, 1.0, 500).astype(np.float32)
combined   = np.concatenate([background, signal])

sigma_old = robust_std_old(combined)
sigma_new = noise_sigma_from_subthreshold(combined)
print(f"\n修复 #7 [OK]  偏态 combined_map 的 sigma 估计（合成数据，真实背景 sigma=1.0）:")
print(f"  旧版全图 MAD sigma : {sigma_old:.4f}")
print(f"  新版子阈值 sigma   : {sigma_new:.4f}  （更接近真实背景噪声）")
# 两者在这个例子里差异不大，因为响应像素只占5%；
# 差异在响应像素比例更高时会更显著（如20%+）
print(f"  注：响应像素占比越高，旧版 sigma 被拉偏越严重")

# ── 验证修复 #8：相对阈值 nostim 判断 ───────────────────────────────
NOSTIM_REL_THRESH = 0.05

def is_nostim(b_max, b_min):
    if b_max <= 0:
        return True
    return (b_max - b_min) / b_max < NOSTIM_REL_THRESH

print("\n修复 #8  相对阈值 nostim 判断（NOSTIM_REL_THRESH=0.05）:")

# 场景1：高增益相机，背景亮度10000，刺激幅度200 → rel=2% < 5% → nostim（信号太弱）
r1 = is_nostim(10000, 9800)
print(f"  高增益，range=200/10000=2%  → is_nostim={r1}  (期望 True，信号弱于阈值)")
assert r1 == True

# 场景2：高增益相机，背景亮度10000，刺激幅度1000 → rel=10% > 5% → 有刺激
r2 = is_nostim(10000, 9000)
print(f"  高增益，range=1000/10000=10% → is_nostim={r2}  (期望 False，有刺激)")
assert r2 == False

# 场景3：低增益相机，背景亮度100，刺激幅度30 → rel=30% > 5% → 有刺激
# 旧版：b_max-b_min=30 < 50 → 错误判为 nostim
# 新版：rel=30% > 5% → 正确判为有刺激
r3 = is_nostim(100, 70)
print(f"  低增益，range=30/100=30%    → is_nostim={r3}  (期望 False；旧版错误判 nostim)")
assert r3 == False

# 场景4：真正无刺激
r4 = is_nostim(100, 98)
print(f"  无刺激，range=2/100=2%      → is_nostim={r4}  (期望 True)")
assert r4 == True

print("\n所有关键修复验证通过 ✓")


修复 #1 [OK]  None sustained_map → 全零数组，不再 AttributeError
修复 #3 [OK]  trial_1 不再错误匹配 trial_10

修复 #7 [OK]  偏态 combined_map 的 sigma 估计（合成数据，真实背景 sigma=1.0）:
  旧版全图 MAD sigma : 1.0544
  新版子阈值 sigma   : 1.0192  （更接近真实背景噪声）
  注：响应像素占比越高，旧版 sigma 被拉偏越严重

修复 #8  相对阈值 nostim 判断（NOSTIM_REL_THRESH=0.05）:
  高增益，range=200/10000=2%  → is_nostim=True  (期望 True，信号弱于阈值)
  高增益，range=1000/10000=10% → is_nostim=False  (期望 False，有刺激)
  低增益，range=30/100=30%    → is_nostim=False  (期望 False；旧版错误判 nostim)
  无刺激，range=2/100=2%      → is_nostim=True  (期望 True)

所有关键修复验证通过 ✓
